In [1]:
import pandas as pd
import wordfreq
import string

In [ ]:
Provo_RT_FILE = "provo_corpus_data/Provo_Corpus-Eyetracking_Data.csv"
Provo_LM_FILE = "provo_corpus_data/Provo_lm_features.csv"
df_RT = pd.read_csv(Provo_RT_FILE)
df_Provo_lm = pd.read_csv(Provo_LM_FILE)
# df_NSC_lm.head(15)

In [3]:
model_names = ['base_gpt2', 'lambda0p0', 'lambda0p001', 'lambda0p01', 'lambda0p1', 'lambda1p0']

def align_features(df, model_names):
    '''
    Align gpt2 tokenization with the tokenization of behavioral data.
    '''
    word_features_dict = {'Word_Unique_ID': df.Word_Unique_ID.iloc[0], 
                          'Text_ID': df.Text_ID.iloc[0],
                          'Sentence_Number': df.Sentence_Number.iloc[0],
                          'Word_Number': df.Word_Number.iloc[0], 
                          'Word_In_Sentence_Number': df.Word_In_Sentence_Number.iloc[0], 
                          'Word': df.Word.iloc[0]}
    for model_name in model_names:
        word_features_dict[f'word_surp_{model_name}'] = df[f'surp_{model_name}'].sum()
    result = pd.DataFrame([word_features_dict])

    return result

# Align gpt2 tokenization with words
df_Provo_lm = df_Provo_lm.groupby("Word_Unique_ID", group_keys=False).apply(
        lambda group: align_features(group, model_names)
        ).reset_index(drop=True)

df_Provo_lm.head(10)

/var/folders/4k/_kj6xmhj7tbbbj61g0kc59280000gn/T/ipykernel_6273/4207799996.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_Provo_lm = df_Provo_lm.groupby("Word_Unique_ID", group_keys=False).apply(


,Word_Unique_ID,Text_ID,Sentence_Number,Word_Number,Word_In_Sentence_Number,Word,word_surp_base_gpt2,word_surp_lambda0p0,word_surp_lambda0p001,word_surp_lambda0p01,word_surp_lambda0p1,word_surp_lambda1p0
0,Q2684,13,4.0,54.0,11.0,carts,18.253006,16.273174,13.966751,16.386560,14.411587,16.951218
1,Q2685,14,2.0,53.0,25.0,stranded,14.599906,15.415355,18.633290,13.930938,12.837726,15.320764
2,Q2686,15,3.0,53.0,19.0,jury,0.012493,0.083531,0.078580,0.010172,0.027085,0.057654
3,Q2688,19,2.0,51.0,20.0,temperature,0.170018,0.033657,1.293276,0.384017,0.345573,0.323223
4,Q2689,24,2.0,48.0,24.0,worth,12.804955,12.525746,12.412022,7.408957,10.868642,8.436168
5,Q2690,25,2.0,48.0,23.0,process,7.077018,5.974581,6.500414,9.280150,7.315862,6.759779
6,Q2691,28,2.0,46.0,28.0,October,6.430339,6.261787,4.042545,4.810759,3.387217,4.331877
7,Q2692,31,2.0,44.0,13.0,pole,18.382868,16.264727,10.615490,13.888976,11.838201,13.597219
8,Q2693,32,2.0,43.0,21.0,mathematics,16.686121,16.111685,19.429678,19.566133,21.298483,17.967245
9,Q2694,33,2.0,43.0,25.0,car,11.600257,13.326685,9.663917,11.889986,11.369226,12.762678


In [4]:
df_RT = df_RT[['Participant_ID', 'Word_Unique_ID', 'Text_ID', 'Word_Number', 'Sentence_Number', 'Word_In_Sentence_Number',
               'Word', 'Word_Length', 'IA_FIRST_FIXATION_DURATION', 'IA_DWELL_TIME']]
df_RT = df_RT.rename(columns={"Participant_ID": "subject",
                              "IA_FIRST_FIXATION_DURATION": "RT_first_fix",
                              "IA_DWELL_TIME": "RT_total"})

df_RT.head()

,subject,Word_Unique_ID,Text_ID,Word_Number,Sentence_Number,Word_In_Sentence_Number,Word,Word_Length,RT_first_fix,RT_total
0,Sub01,QID1,1,2.0,1.0,2.0,are,3.0,147.0,147
1,Sub01,QID2,1,3.0,1.0,3.0,now,3.0,193.0,193
2,Sub01,QID3,1,4.0,1.0,4.0,rumblings,9.0,198.0,198
3,Sub01,QID4,1,5.0,1.0,5.0,that,4.0,NaN,0
4,Sub01,QID5,1,6.0,1.0,6.0,Apple,5.0,235.0,235


In [5]:
result_df = pd.merge(df_RT, df_Provo_lm, how='inner', on=['Word_Unique_ID', 'Text_ID', 'Word_Number', 'Sentence_Number', 'Word_In_Sentence_Number'])
print(df_RT.shape)
print(result_df.shape)
result_df = result_df.rename(columns={"Word_x": "Word", 
                                      "Word_y": "Word_lm"})
result_df.head()

(230412, 10)
(223272, 17)


,subject,Word_Unique_ID,Text_ID,Word_Number,Sentence_Number,Word_In_Sentence_Number,Word,Word_Length,RT_first_fix,RT_total,Word_lm,word_surp_base_gpt2,word_surp_lambda0p0,word_surp_lambda0p001,word_surp_lambda0p01,word_surp_lambda0p1,word_surp_lambda1p0
0,Sub01,QID1,1,2.0,1.0,2.0,are,3.0,147.0,147,are,1.657564,1.879504,1.882209,4.977292,4.188390,3.493596
1,Sub01,QID2,1,3.0,1.0,3.0,now,3.0,193.0,193,now,7.695737,7.533978,7.328161,7.350562,7.327811,8.157033
2,Sub01,QID3,1,4.0,1.0,4.0,rumblings,9.0,198.0,198,rumblings,17.577855,25.342977,26.967551,24.038426,24.759908,26.484137
3,Sub01,QID4,1,5.0,1.0,5.0,that,4.0,NaN,0,that,4.944870,5.016036,3.772242,4.869665,5.475588,4.234495
4,Sub01,QID5,1,6.0,1.0,6.0,Apple,5.0,235.0,235,Apple,17.096480,19.961027,18.118113,17.816225,18.473840,19.112215


In [6]:
# Add word frequency
def get_frequency(word, language='en'):
	return wordfreq.zipf_frequency(word, language)

result_df['logfreq'] = result_df.apply(
	lambda df: get_frequency(df['Word'], 'en'), axis=1
	)

In [7]:
# Mark punctuation
result_df['has_punct'] = result_df['Word'].str.contains(rf"[{string.punctuation}]", regex=True)

In [8]:
result_df.to_csv('corpus_data/Provo_RT_lm_features.csv', index=False)